<a href="https://colab.research.google.com/github/ShadowMesh001/Email-Spam-Detection-Using-Machine-Learning/blob/main/spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Email Spam Detection using SVM & Naive Bayes
In this notebook, we will:

Load a sample Email/SMS Spam dataset.
Preprocess the text data (tokenization, vectorization using TF-IDF).
Train a Naive Bayes classifier.
Train a Support Vector Machine (SVM) classifier.
Evaluate both models using Accuracy, Precision, Recall, and F1-Score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import urllib.request
import zipfile
import os

# Download the UCI SMS Spam Collection dataset
url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
zip_path = 'sms_spam_collection.zip'
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('sms_spam')

# Load dataset
df = pd.read_csv('sms_spam/SMSSpamCollection', sep='\t', names=['label', 'message'])
display(df.head())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Data Preprocessing and Splitting

In [ ]:
# Convert labels to binary values (spam = 1, ham = 0)
df['label'] = df['label'].map({'spam': 1, 'ham': 0})

X = df['message']
y = df['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training set shape: {X_train_tfidf.shape}")
print(f"Testing set shape: {X_test_tfidf.shape}")

Training set shape: (4457, 5000)
Testing set shape: (1115, 5000)


Model Training & Evaluation


In [ ]:
# Initialize models
nb_model = MultinomialNB()
svm_model = SVC(kernel='linear', C=1.0, random_state=42)

# Train Naive Bayes
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

# Train SVM
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)

# Function to evaluate performance
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"=== {model_name} Evaluation ===")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")

    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1}

# Run evaluation
nb_results = evaluate_model(y_test, nb_preds, "Naive Bayes")
svm_results = evaluate_model(y_test, svm_preds, "Support Vector Machine (SVM)")

=== Naive Bayes Evaluation ===
Accuracy:  0.9722
Precision: 1.0000
Recall:    0.7919
F1-Score:  0.8839

=== Support Vector Machine (SVM) Evaluation ===
Accuracy:  0.9821
Precision: 0.9850
Recall:    0.8792
F1-Score:  0.9291



Model Comparison Summary

In [ ]:
summary_df = pd.DataFrame([nb_results, svm_results], index=['Naive Bayes', 'SVM'])
display(summary_df)

,Accuracy,Precision,Recall,F1-Score
Naive Bayes,0.972197,1.000000,0.791946,0.883895
SVM,0.982063,0.984962,0.879195,0.929078
